In [0]:
import dlt
from pyspark import sql
from pyspark.sql import functions as F
from pyspark.sql.types import *

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-5868425954704765>, line 1
----> 1 import dlt
      2 from pyspark import sql
      3 from pyspark.sql import functions as F

ModuleNotFoundError: No module named 'dlt'

In [0]:
dataset_rules = {

    "R1": "Employee_ID IS NOT NULL",                # Every record must have an ID

    "R2": "Name IS NOT NULL AND length(Name) > 0",  # Name cannot be blank

    "R3": "Department IS NOT NULL",                 # Department mandatory

    "R4": "Performance_Score IS NOT NULL",          # Score cannot be missing

    "R5": "Monthly_Salary IS NOT NULL AND Monthly_Salary > 0"  # Salary must be positive

}
 

In [0]:
@dlt.table(
    comment="Bronze layer: read employee data with Change Data Feed enabled"
)
@dlt.expect_all(dataset_rules)
def bronze_employee_cdf():
    '''
    Reads from the raw_employee_cdf Delta table with Change Data Feed (CDF) enabled.
    Captures inserts, updates, and deletes automatically.
    '''
    cdf = (
        spark.readStream
             .format("delta")
             .option("readChangeFeed", "true")     # enable CDC
             .table("workspace.damg7370.sample_employee_cdf")  # must have CDC enabled
             # .where("_change_type != 'delete'")  # optional: skip deletes
    )
 
    return cdf